# Building the dataset

How the raw material of the whole project was assembled: APK samples are collected by their SHA256 hashes, placed into the Drebin folder layout of the `malware_detection` codebase, turned into feature vectors, and saved as the PyTorch datasets every detector in `02-detectors/` trains on.

Four parts: **A** — download and assemble the samples; **B** — import the extracted features and build the train/validation/test datasets; **C** — updating the dataset later with new samples; **D** — the feature-extraction runs themselves.

*Consolidated from `create_inputs.ipynb`, `import_features.ipynb`, `update_datasets.ipynb`, `update_datasets2.ipynb`, `feature_extraction.ipynb` and `future_extraction.ipynb` (originally in `a_malware` / `a_malware2`). The originals interleaved this pipeline with git housekeeping and repeated downloads; those cells are removed and each step appears once. Outputs cleared.*

## Part A — Download and assemble the samples

### Step 1 — The sample lists (benign and malicious SHA256 hashes)

In [ ]:
# our sharable address : https://drive.google.com/file/d/1JFZY6bOua4XvL1t36nNF5hTigotYckIY/view?usp=share_link
# Replace 'your_file_id' with the actual file ID from the shareable link
file_id = '1JFZY6bOua4XvL1t36nNF5hTigotYckIY'

# Create a direct download link
download_link = f'https://drive.google.com/uc?id={file_id}'

# Use wget to download the file
!wget "$download_link" -O benign_sha256.csv

In [ ]:
# Replace 'your_file_id' with the actual file ID from the shareable link
file_id = '1veh1yx5JzxziRG6tYfBOKnfKi2c5u7Mv'

# Create a direct download link
download_link = f'https://drive.google.com/uc?id={file_id}'

# Use wget to download the file
!wget "$download_link" -O malicious_sha256.csv

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df_benign = pd.read_csv('/content/benign_sha256.csv')

df_benign.head(3)

In [ ]:
# Extract the first column as a list
benign_sha256 = df_benign.iloc[:, 0].tolist()[:1000]
benign_sha256[:3]

In [ ]:
# Read the CSV file into a DataFrame
df_malicious = pd.read_csv('/content/malicious_sha256.csv')

df_malicious.head(3)

In [ ]:
# Extract the first column as a list
malicious_sha256 = df_malicious.iloc[:, 0].tolist()[:1000]
malicious_sha256[:3]

### Step 2 — Download the samples

In [ ]:
# download benign samples

import os
import requests
import concurrent.futures

# Replace with your actual API key in androidzoo
APIKEY = "717edc01a24bab5afe87835503feb736540c51352fe02bf2e487146caf0fa192"

# List of SHA256 hashes
sha256_list = benign_sha256

# Base URL for the download
base_url = "https://androzoo.uni.lu/api/download"

# Destination directory for downloaded APKs
download_dir = "/content/datasets/benign/"

# Create the download directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

# Function to download an APK file
def download_apk(sha256):
    params = {"apikey": APIKEY, "sha256": sha256}
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        filename = sha256
        file_path = os.path.join(download_dir, filename)
        with open(file_path, "wb") as apk_file:
            apk_file.write(response.content)
        print(f"Downloaded: {filename}")
    else:
        print(f"Failed to download APK with SHA256: {sha256}")

# Set the maximum number of concurrent downloads (adjust as needed)
max_concurrent_downloads = 40

# Use ThreadPoolExecutor to run download_apk concurrently
with concurrent.futures.ThreadPoolExecutor(max_concurrent_downloads) as executor:
    executor.map(download_apk, sha256_list)

print("Download process completed.")

In [ ]:
## download malicious samples

# List of SHA256 hashes
sha256_list = malicious_sha256

# Base URL for the download
base_url = "https://androzoo.uni.lu/api/download"

# Destination directory for downloaded APKs
download_dir = "/content/datasets/malicious/"

# Create the download directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

# Function to download an APK file
def download_apk(sha256):
    params = {"apikey": APIKEY, "sha256": sha256}
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        filename = sha256
        file_path = os.path.join(download_dir, filename)
        with open(file_path, "wb") as apk_file:
            apk_file.write(response.content)
        print(f"Downloaded: {filename}")
    else:
        print(f"Failed to download APK with SHA256: {sha256}")

# Set the maximum number of concurrent downloads (adjust as needed)
max_concurrent_downloads = 40

# Use ThreadPoolExecutor to run download_apk concurrently
with concurrent.futures.ThreadPoolExecutor(max_concurrent_downloads) as executor:
    executor.map(download_apk, sha256_list)

print("Download process completed.")

### Step 3 — Bring in the codebase and place the samples into its Drebin layout

In [ ]:
!unzip '/content/uc?id=1rXloO77nNsjSmdbVlcb0fX1-H4tKBNbp'

In [ ]:
!git clone https://github.com/mostafa-ja/malware_detection.git

In [ ]:
!mv /content/datasets/benign/* /content/malware_detection/datasets/drebin/benign_samples/
!mv /content/datasets/malicious/* /content/malware_detection/datasets/drebin/malicious_samples/
!mv /content/naive_data /content/malware_detection/datasets/naive_data

In [ ]:
import os
folder_path = '/content/malware_detection/datasets/drebin/benign_samples'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in benign_samples : {file_count}")

In [ ]:
import os
folder_path = '/content/malware_detection/datasets/drebin/malicious_samples'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in malicious_samples : {file_count}")

In [ ]:
import os
folder_path = '/content/malware_detection/datasets/naive_data'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in malicious_samples : {file_count}")

In [ ]:
import pickle

# Open and load the Pickle file
with open('/content/malware_detection/datasets/naive_data/629e1bc6a81d8b717dbcc0a942e60036ea61baf134e013b58bfdd2b2c06eeac7.feat', 'rb') as file:
    loaded_data = pickle.load(file)

# Access and use the loaded data
print(loaded_data)  # Example: Print the loaded data

In [ ]:
len(loaded_data)

## Part B — From extracted features to PyTorch datasets

### Step 4 — Load the extracted feature data

In [ ]:
!pip install androguard
import androguard

In [ ]:
import sys
sys.path.append('/content/malware_detection/')

import dataset

In [ ]:
# Dataset is a class
data = dataset.Dataset()
data.vocab_size

### Step 5 — Build the train / validation / test datasets

In [ ]:
train_paths = data.train_dataset
train_paths #([paths],[labels])

In [ ]:
train_samples = dataset.DatasetTorch(*train_paths, data, name='train', use_cache=False)
train_samples

In [ ]:
for sample in train_samples:
  print(sample)
  break

In [ ]:
sample[0].shape

In [ ]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = [(torch.tensor(data, dtype=torch.float32), label) for data, label in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

train_dataset = MyDataset(train_samples)

In [ ]:
# Save the dataset to a file in PyTorch's native format
torch.save(train_dataset, "train_dataset.pt")

In [ ]:
validation_paths = data.validation_dataset #([paths],[labels])

validation_samples = dataset.DatasetTorch(*validation_paths, data, name='val', use_cache=False)

validation_dataset = MyDataset(validation_samples)

torch.save(validation_dataset, "validation_dataset.pt")

In [ ]:
test_paths = data.test_dataset #([paths],[labels])

test_samples = dataset.DatasetTorch(*test_paths, data, name='test', use_cache=False)

test_dataset = MyDataset(test_samples)

torch.save(test_dataset, "test_dataset.pt")

### Step 6 — Inspect what was built (vocabulary and sample vectors)

In [ ]:
len(data.vocab)

In [ ]:
data.vocab[:3]

In [ ]:
sample_list = ['android.permission.WRITE_EXTERNAL_STORAGE',
               'android.permission.READ_EXTERNAL_STORAGE',
               'android.permission.PROCESS_OUTGOING_CALLS',
               'android.permission.READ_PHONE_STATE',
               'android.permission.ACCESS_COARSE_LOCATION',
               'android.permission.ACCESS_FINE_LOCATION',
               'android.permission.WRITE_CALENDAR',
               'android.permission.READ_CALENDAR',
               'android.permission.GET_ACCOUNTS',
               'android.permission.RECORD_AUDIO',
               'android.permission.CAMERA',
               'android.permission.CALL_PHONE',
               'android.permission.READ_CONTACTS',
               'android.permission.READ_CALL_LOG',
               'android.permission.SEND_SMS',
               'android.permission.WRITE_CONTACTS',
               'android.permission.RECEIVE_SMS',
               'android.permission.READ_SMS',
               'android.permission.WRITE_CALL_LOG',
               'android.intent.action.MAIN',
               'android.intent.action.BOOT_COMPLETED',
               'com.android.vending.INSTALL_REFERRER',
               'android.intent.action.VIEW',
               'android.intent.action.Run',
               'android.net.conn.CONNECTIVITY_CHANGE',
               'android.intent.action.LOCALE_CHANGED',
               'android.intent.action.USER_PRESENT',
               'android.intent.action.PACKAGE_ADDED',
               'android.intent.action.PACKAGE_REMOVED']

target_values = ['android.permission.READ_EXTERNAL_STORAGE','android.net.conn.CONNECTIVITY_CHANGE', 'android.permission.READ_CONTACTS', 'android.intent.action.PACKAGE_ADDED']

# Using a list comprehension to find the indices (args)
indices = [sample_list.index(target) if target in sample_list else -1 for target in target_values]

print(indices)

test and val set are not shuffled

In [ ]:
import pickle

# Open and load the Pickle file
with open(test_paths[0][0], 'rb') as file:
    loaded_data = pickle.load(file)

# Access and use the loaded data
print(loaded_data)  # Example: Print the loaded data

In [ ]:
target_values = ['android.permission.ACCESS_NETWORK_STATE','android.permission.INTERNET','tdedlomtoe.ballmaen.com.MainActivity']
indices = [data.vocab.index(target) if target in data.vocab else -1 for target in target_values]

print(indices)

In [ ]:
for i,j in test_samples:
  print(i)
  break

In [ ]:
i[654]

In [ ]:
i[706]

## Part C — Updating the dataset later

New samples were added in batches (Part A repeated for a slice of the lists), then features were re-extracted and the result copied to Drive as the new naive_data.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
start_index = 150
end_index = 152

In [ ]:
!python -m examples.feature_extraction_test  --proc_number 20 --number_of_smali_files 1000000 --max_vocab_size 10000

In [ ]:
!python -m examples.feature_extraction_test  --proc_number 10 --number_of_smali_files 1000000 --max_vocab_size 10000

In [ ]:
!cp -r  /content/malware_detection/datasets/naive_data /content/drive/MyDrive/

In [ ]:
folder_path = '/content/drive/MyDrive/naive_data'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in naive_data : {file_count}")

## Part D — The feature-extraction runs

Androguard and Captum are the codebase's extraction dependencies; the extraction itself runs through the codebase's own test entry points. The last two cells show the second-generation codebase (`malware_detection2`) that the same pipeline was later pointed at.

In [ ]:
!pip install captum
import captum

In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
import argparse

from core.droidfeature import Apk2features
from config import config

In [ ]:
malware_dir_name = '/content/pad4amd_master/csbd-master/data/small_proto_apks/malware'
benware_dir_name = '/content/pad4amd_master/csbd-master/data/small_proto_apks/goodware'
meta_data_saving_dir = '/content/'
naive_data_saving_dir = '/content/'
number_of_smali_files = 1000000
max_vocab_size = 10000
update = False
proc_number = 4

feature_extractor = Apk2features(naive_data_saving_dir,
                                  meta_data_saving_dir,
                                  number_of_smali_files=number_of_smali_files,
                                  max_vocab_size=max_vocab_size,
                                  update=update,
                                  proc_number=proc_number)
malware_features = feature_extractor.feature_extraction(malware_dir_name)
print('The number of malware files: ', len(malware_features))
benign_features = feature_extractor.feature_extraction(benware_dir_name)
print('The number of benign files: ', len(benign_features))

In [ ]:
malware_features

In [ ]:
!python -m examples.feature_extraction_test --update --proc_number 10 --number_of_smali_files 1000000 --max_vocab_size 10000

In [ ]:
!python -m examples.md_nn_test --cuda --cache --seed 0 --batch_size 128 --proc_number 10 --epochs 50 --max_vocab_size 10000 --dense_hidden_units "200,200" --weight_decay 0.0 --lr 0.001 --dropout 0.6

In [ ]:
!git clone https://github.com/mostafa-ja/malware_detection2.git

In [ ]:
%cd /content/malware_detection2